In [1]:
import pandas as pd
import yaml

from internalizer import Internalizer
from internalizer.calculation_setup import *
from internalizer.regionalization import REMIND_REGIONS
from internalizer.utils import check_monetization_factors
import time

In [2]:
from internalizer.internalizer import CONFIG_NO_REMOVAL

In [3]:
EI_VERSION = "3.10"

mifpath = "/p/tmp/davidba/internalization_develop/remind/output/SSP2-NPi-internalize-combined_2025-09-19_11.34.29/lca/remind_runs/remind_SSP2-NPi-internalize-combined.mif"
gdxpath = "/p/tmp/davidba/internalization_develop/remind/output/SSP2-NPi-internalize-combined_2025-09-19_11.34.29/input.gdx"
pathway = "SSP2-NPi-internalize-combined"
monetization = 0.5
# monetization = {
#     "ReCiPe 2016 v1.03, midpoint (H) - acidification: terrestrial - terrestrial acidification potential (TAP)" : 5.27,
#   #   "ReCiPe 2016 v1.03, midpoint (H) - ecotoxicity: terrestrial - terrestrial ecotoxicity potential (TETP)": 6.4e-05
# }
plca = False
calcCosts = True
aggTaxes = True
# years = [2020, 2030, 2040, 2050]
years = [2020, 2030, 2050]
yaml_file = "../internalizer/data/mappings/remind_internalization_setup_v2.yaml"
levels = "SE,FE"


# set up brightway project
bw_project = f"internalizer_ei_{EI_VERSION}"

In [4]:
IMPACT_CATEGORIES_MC = [
    "acidification",
    "climate change",
    "ecotoxicity",
    "eutrophication",
    "fossil resources",
    "human toxicity",
    "ionizing radiation",
    "land use",
    "metal/mineral resources",
    "ozone depletion",
    "particulate matter formation",
    "photochemical oxidant formation",
    "water use"
]

In [5]:
# initialize Internalizer
I = Internalizer(
    mifpath,
    "remind",
    pathway,
    EI_VERSION,
    bw_project,
    gdxpath,
    outputfolder = "lca_v2"
)
print(I.scenario)

SSP2-NPi-internalize-combined


In [16]:

if plca:
    t0 = time.time()
    # add_ES_subcategories(mifpath)
    I.run_premise(years)
    t1 = time.time()
    print(f"Premise runs done in {t1-t0} seconds", "\n")

else:
    I.years = years

I.set_calculation_setup(levels=levels) # defaults to REMIND Internalization setup
print("Calculation setup set", "\n")

if calcCosts:
    t0 = time.time()
    I.calculate_costs(monetization, save_intermediate_results=True)
    t1 = time.time()
    print(f"Cost calculation done in {t1-t0} seconds", "\n")

if aggTaxes:
    t0 = time.time()
    I.load_costs()

    ics = IMPACT_CATEGORIES_MC
    if isinstance(monetization, dict):
        ics = list(monetization.keys())
    I.write_remind_input_files(
        2020,
        2030,
        ics
    )
    t1 = time.time()
    print(f"Tax recalculation done in {t1-t0} seconds", "\n")


Calculation setup set 

se2h2, 2020: Matrices loaded
se2h2, 2020: Functional units selected
pe2se, 2050: Matrices loaded
pe2se, 2050: Functional units selected
pe2se, 2050: 0 activities in the removal list
pe2se, 2020: Matrices loaded
pe2se, 2020: Functional units selected
pe2se, 2020: 0 activities in the removal list
pe2se, 2030: Matrices loaded
pe2se, 2030: Functional units selected
pe2se, 2030: 0 activities in the removal list
se2h2, 2020: 4446 activities in the removal list
se2h2, 2020: LCI done.
se2h2, 2020: Cost calculation done.
se2h2, 2020: Regionalization done.
se2h2, 2030: Matrices loaded
se2h2, 2030: Functional units selected
se2h2, 2030: 4446 activities in the removal list
se2h2, 2030: LCI done.
se2h2, 2030: Cost calculation done.
se2h2, 2030: Regionalization done.
se2h2, 2050: Matrices loaded
se2h2, 2050: Functional units selected
se2h2, 2050: 4446 activities in the removal list
se2h2, 2050: LCI done.
se2h2, 2050: Cost calculation done.
se2h2, 2050: Regionalization done.
h

In [13]:
I.load_costs()

In [14]:
I

In [9]:
I.cs.levels

['pe2se', 'se2h2', 'h22se', 'fe']

In [7]:
I.cs.data["fe"]["removal list"]

,dataset name,dataset reference product,dataset unit
0,"biodiesel production, via Fischer-Tropsch, fro...","biodiesel, from forest residues",kilogram
1,"biodiesel production, via Fischer-Tropsch, fro...","biodiesel, from forest residues",kilogram
2,"biodiesel production, via transesterification,...","biodiesel, from algae",kilogram
3,"biodiesel production, via transesterification,...","biodiesel, from palm oil",kilogram
4,"biodiesel production, via transesterification,...","biodiesel, from rapeseed oil",kilogram
...,...,...,...
730,"wood chips production, hardwood, at sawmill","wood chips, wet, measured as dry mass",kilogram
731,"wood chips production, softwood, at sawmill","wood chips, wet, measured as dry mass",kilogram
732,wood pellet production,"wood pellet, measured as dry mass",kilogram
733,"wood pellets, burned in stirling heat and powe...","electricity, low voltage",kilowatt hour


In [28]:
# load raw costs and aggregate with mappings
I.cost_results = {}
for lvl in I.cs.levels:
    I.cost_results[lvl] = {}
    for year in I.years:
        # recalculate mapping
        mapping = I.cs.data[lvl].get(
            "regionalized mapping", I.cs.regionalize_dynamic_mapping(lvl, year)
        )
        fp = I.outdir + f"/{I.model}/{I.scenario}/{str(year)}/regionalized_costs_{lvl}.csv"
        regionalized_costs = pd.read_csv(fp)
        costs_agg = combine_shares_and_costs(mapping, regionalized_costs).melt(
            var_name="impact category", value_name="cost", ignore_index=False
        ).reset_index()
        I.cost_results[lvl][year] = costs_agg.set_index(
            ["REMIND index", "region", "impact category"]
        )["cost"].to_xarray()

In [12]:
mapping = I.cs.data["fe"]["base mapping"]


# extend mapping
dflist = []
for region in REMIND_REGIONS:
    df = mapping.copy()
    df["region"] = region
    dflist.append(df)

regionalized_mapping = pd.concat(dflist).set_index(["scenario variable", "region"])

# load .mif
mif = pd.read_csv(mifpath, sep=";").rename(columns={"Region": "region", "Variable": "scenario variable"})
mifdata = mif.set_index(["scenario variable", "region"])[str(year)]

# select shared index (some regions don't have certain FE variables)
combined_idx = regionalized_mapping.index.intersection(mifdata.index)
sel = regionalized_mapping.loc[combined_idx]

# calculate shares
sel["weight"] = mifdata.loc[combined_idx]
sel = sel.reset_index()
sel["total"] = sel.reset_index().groupby(["REMIND index", "region"])["weight"].transform("sum")

# where total is zero, set weights to one and recalculate total
def robust_weights(row):
    if row["total"] == 0:
        return 1
    else:
        return row["weight"]
sel["weight new"] = sel.apply(robust_weights, axis=1)
sel["total new"] = sel.groupby(["REMIND index", "region"])["weight new"].transform("sum")
sel["share"] = sel["weight new"] / sel["total new"]


In [14]:
sel[sel["total"] == 0]

,scenario variable,region,REMIND index,dataset name,dataset reference product,dataset unit,share,weight,total,weight new,total new
4,FE|Buildings|Heating|Hydrogen,CAZ,build - feh2s,"heat, residential, by combustion of hydrogen u...","heat, from residential heating system",megajoule,1.000000,0.0,0.0,1.0,1.0
15,FE|CDR|DAC|+|Gases,CAZ,cdr - fegas,"heat production, natural gas, at boiler conden...","heat, district or industrial, natural gas",megajoule,0.500000,0.0,0.0,1.0,2.0
17,FE|CDR|DAC|+|Hydrogen,CAZ,cdr - feh2s,"heat production, from hydrogen-fired one gigaw...",heat,megajoule,0.333333,0.0,0.0,1.0,3.0
18,FE|CDR|EW|+|Diesel,CAZ,cdr - fedie,"diesel, burned in diesel-electric generating s...","diesel, burned in diesel-electric generating s...",megajoule,0.333333,0.0,0.0,1.0,3.0
20,"FE|CDR|OAE, electric calciner|+|Diesel",CAZ,cdr - fedie,"diesel, burned in diesel-electric generating s...","diesel, burned in diesel-electric generating s...",megajoule,0.333333,0.0,0.0,1.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...
2237,"FE|CDR|OAE, electric calciner|+|Diesel",USA,cdr - fedie,"diesel, burned in diesel-electric generating s...","diesel, burned in diesel-electric generating s...",megajoule,0.333333,0.0,0.0,1.0,3.0
2239,"FE|CDR|OAE, electric calciner|+|Hydrogen",USA,cdr - feh2s,"hydrogen, burned in gas turbine 1GW",heat,megajoule,0.333333,0.0,0.0,1.0,3.0
2240,"FE|CDR|OAE, traditional calciner|+|Diesel",USA,cdr - fedie,"diesel, burned in diesel-electric generating s...","diesel, burned in diesel-electric generating s...",megajoule,0.333333,0.0,0.0,1.0,3.0
2242,"FE|CDR|OAE, traditional calciner|+|Gases",USA,cdr - fegas,"natural gas, burned in gas turbine","natural gas, burned in gas turbine",megajoule,0.500000,0.0,0.0,1.0,2.0


In [9]:
lvl = "fe"
year = 2030
mapping = I.cs.data[lvl].get(
    "regionalized mapping", I.cs.regionalize_dynamic_mapping(lvl, year)
)

In [10]:
mapping

,REMIND index,dataset name,dataset reference product,dataset unit,share,region
0,build - fehes,"market for heat, district or industrial, natur...","heat, district or industrial, natural gas",megajoule,1.000000,CAZ
1,build - feels,"heat production, air-water heat pump 10kW","heat, air-water heat pump 10kW",megajoule,0.067934,CAZ
2,build - feels,"heat, residential, electric storage heater, us...","heat, from residential heating system",megajoule,0.225984,CAZ
3,build - fegas,"heat production, natural gas, at boiler conden...","heat, central or small-scale, natural gas",megajoule,1.000000,CAZ
4,build - feh2s,"heat, residential, by combustion of hydrogen u...","heat, from residential heating system",megajoule,1.000000,CAZ
...,...,...,...,...,...,...
2395,trans - fedie,"diesel, synthetic, burned in passenger car",heat,megajoule,0.000000,USA
2396,trans - feelt,"electricity, used in battery electric motorcycle","electricity, low voltage",megajoule,0.014162,USA
2397,trans - fepet,"bioethanol, burned in motorcycle",heat,megajoule,0.000094,USA
2398,trans - fepet,"petrol, burned in motorcycle",heat,megajoule,0.002531,USA


In [30]:
regionalized_costs = regionalized_costs.pivot(
    index=["dataset name", "dataset reference product", "dataset unit", "region"],
    columns="impact category",
    values="cost"
)

In [31]:
regionalized_costs

impact category                                                                                   acidification  \
dataset name                                       dataset reference product dataset unit region                  
biodiesel, burned in heavy-duty vehicle            heat                      megajoule    CAZ          0.000239   
                                                                                          CHA          0.000257   
                                                                                          EUR          0.000245   
                                                                                          IND          0.000241   
                                                                                          JPN          0.000263   
...                                                                                                         ...   
soft wood chips from forest, burned in furnace ... heat                      megajoule    OAS          0.000221   
                                                                                          REF          0.000221   
                                                                                          SSA          0.000221   
                                                                                          USA          0.000221   
                                                                                          World        0.000221   

impact category                                                                                   climate change  \
dataset name                                       dataset reference product dataset unit region                   
biodiesel, burned in heavy-duty vehicle            heat                      megajoule    CAZ           0.001880   
                                                                                          CHA           0.002326   
                                                                                          EUR           0.001781   
                                                                                          IND           0.001926   
                                                                                          JPN           0.002016   
...                                                                                                          ...   
soft wood chips from forest, burned in furnace ... heat                      megajoule    OAS           0.002214   
                                                                                          REF           0.002214   
                                                                                          SSA           0.002214   
                                                                                          USA           0.002214   
                                                                                          World         0.002214   

impact category                                                                                   ecotoxicity  \
dataset name                                       dataset reference product dataset unit region                
biodiesel, burned in heavy-duty vehicle            heat                      megajoule    CAZ        0.000840   
                                                                                          CHA        0.001096   
                                                                                          EUR        0.000812   
                                                                                          IND        0.000880   
                                                                                          JPN        0.001059   
...                                                                                                       ...   
soft wood chips from forest, burned in furnace ... heat                      megajoule    O

In [32]:
mapping = mapping.set_index(["dataset name", "dataset reference product", "dataset unit", "region"])

In [ ]:
old_idx = mapping.index
new_idx = []
for idx in old_idx:
    if idx in regionalized_costs.index:
        new_idx.append(idx)
    else:
        new_idx.append((idx[0], idx[1], idx[2], "World"))

In [ ]:
combined = pd.DataFrame(
    regionalized_costs.loc[new_idx].to_numpy(),
    index=old_idx,
    columns=regionalized_costs.columns
).mul(mapping["share"], axis=0)

In [ ]:
combined["REMIND index"] = mapping["REMIND index"]
combined

impact category                                                                                     ReCiPe 2016 v1.03, midpoint (H) - acidification: terrestrial - terrestrial acidification potential (TAP)  \
dataset name                                       dataset reference product  dataset unit  region                                                                                                             
electricity production, natural gas, combined c... electricity, high voltage  kilowatt hour CAZ                                              0.000213                                                          
electricity production, at natural gas-fired co... electricity, high voltage  kilowatt hour CAZ                                              0.000271                                                          
electricity production, natural gas, convention... electricity, high voltage  kilowatt hour CAZ                                              0.000496                                                          
petroleum and gas production, onshore              natural gas, high pressure cubic meter   CAZ                                              0.000034                                                          
petroleum and gas production, offshore             natural gas, high pressure cubic meter   CAZ                                              0.000011                                                          
...                                                                                                                                               ...                                                          
ethanol production, via fermentation, from misc... ethanol, from miscanthus   kilogram      LAM                                              0.000727                                                          
                                                                                            MEA                                              0.000894                                                          
                                                                                            OAS                                              0.000844                                                          
                                                                                            SSA                                              0.000981                                                          
                                                                                            USA                                              0.000310                                                          

impact category                                                                                    REMIND index  
dataset name                                       dataset reference product  dataset unit  region               
electricity production, natural gas, combined c... electricity, high voltage  kilowatt hour CAZ            ngcc  
electricity production, at natural gas-fired co... electricity, high voltage  kilowatt hour CAZ           ngccc  
electricity production, natural gas, convention... electricity, high voltage  kilowatt hour CAZ             ngt  
petroleum and gas production, onshore              natural gas, high pressure cubic meter   CAZ           gastr  
petroleum and gas production, offshore             natural gas, high pressure cubic meter   CAZ           gastr  
...                                                                                                         ...  
ethanol production, via fermentation, from misc... ethanol, from miscanthus   kilogram      LAM         bioethl  
                                                                                            MEA         bioethl  
                                                                                            OAS         bioethl  
                                         

In [ ]:
combined.reset_index().groupby(["REMIND index", "region"])[regionalized_costs.columns].sum()

impact category      ReCiPe 2016 v1.03, midpoint (H) - acidification: terrestrial - terrestrial acidification potential (TAP)
REMIND index region                                                                                                          
MeOH         CAZ                                              0.002039                                                       
             CHA                                              0.002040                                                       
             EUR                                              0.002040                                                       
             IND                                              0.002040                                                       
             JPN                                              0.002040                                                       
...                                                                ...                                                       
windoff      NEU                                              0.000142                                                       
             OAS                                              0.000142                                                       
             REF                                              0.000142                                                       
             SSA                                              0.000142                                                       
             USA                                              0.000142                                                       

[624 rows x 1 columns]

In [ ]:
mapping.share

dataset name                                                                                                  dataset reference product   dataset unit   region
electricity production, natural gas, combined cycle power plant                                               electricity, high voltage   kilowatt hour  CAZ       1.0
electricity production, at natural gas-fired combined cycle power plant, post, pipeline 200km, storage 1000m  electricity, high voltage   kilowatt hour  CAZ       1.0
electricity production, natural gas, conventional power plant                                                 electricity, high voltage   kilowatt hour  CAZ       1.0
petroleum and gas production, onshore                                                                         natural gas, high pressure  cubic meter    CAZ       0.7
petroleum and gas production, offshore                                                                        natural gas, high pressure  cubic meter    CAZ       0.3
     

In [ ]:
combined.mul(mapping["share"], axis=0)

impact category                                                                                     ReCiPe 2016 v1.03, midpoint (H) - acidification: terrestrial - terrestrial acidification potential (TAP)
dataset name                                       dataset reference product  dataset unit  region                                                                                                          
electricity production, natural gas, combined c... electricity, high voltage  kilowatt hour CAZ                                              0.000213                                                       
electricity production, at natural gas-fired co... electricity, high voltage  kilowatt hour CAZ                                              0.000271                                                       
electricity production, natural gas, convention... electricity, high voltage  kilowatt hour CAZ                                              0.000496                                                       
petroleum and gas production, onshore              natural gas, high pressure cubic meter   CAZ                                              0.000034                                                       
petroleum and gas production, offshore             natural gas, high pressure cubic meter   CAZ                                              0.000011                                                       
...                                                                                                                                               ...                                                       
ethanol production, via fermentation, from misc... ethanol, from miscanthus   kilogram      LAM                                              0.000727                                                       
                                                                                            MEA                                              0.000894                                                       
                                                                                            OAS                                              0.000844                                                       
                                                                                            SSA                                              0.000981                                                       
                                                                                            USA                                              0.000310                                                       

[988 rows x 1 columns]

In [ ]:
global_idx = mapping.index.difference(regionalized_costs.index)
idx_frame = global_idx.to_frame().set_index(np.arange(len(global_idx)))
idx_frame["region"] = "World"

new_index = pd.MultiIndex.from_frame(idx_frame).union(mapping.index.intersection(regionalized_costs.index))
new_index

MultiIndex([(    'biodiesel production, via transesterification, from palm oil, energy allocation', ...),
            (    'biodiesel production, via transesterification, from palm oil, energy allocation', ...),
            (    'biodiesel production, via transesterification, from palm oil, energy allocation', ...),
            (    'biodiesel production, via transesterification, from palm oil, energy allocation', ...),
            (    'biodiesel production, via transesterification, from palm oil, energy allocation', ...),
            ('biodiesel production, via transesterification, from rapeseed oil, energy allocation', ...),
            ('biodiesel production, via transesterification, from rapeseed oil, energy allocation', ...),
            ('biodiesel production, via transesterification, from rapeseed oil, energy allocation', ...),
            ('biodiesel production, via transesterification, from rapeseed oil, energy allocation', ...),
            ('biodiesel production, via transe

In [ ]:
mapping["share"]

dataset name                                                                                                  dataset reference product   dataset unit   region
electricity production, natural gas, combined cycle power plant                                               electricity, high voltage   kilowatt hour  CAZ       1.0
electricity production, at natural gas-fired combined cycle power plant, post, pipeline 200km, storage 1000m  electricity, high voltage   kilowatt hour  CAZ       1.0
electricity production, natural gas, conventional power plant                                                 electricity, high voltage   kilowatt hour  CAZ       1.0
petroleum and gas production, onshore                                                                         natural gas, high pressure  cubic meter    CAZ       0.7
petroleum and gas production, offshore                                                                        natural gas, high pressure  cubic meter    CAZ       0.3
     

In [ ]:
regionalized_costs.loc[new_index]

impact category                                                                                           ReCiPe 2016 v1.03, midpoint (H) - acidification: terrestrial - terrestrial acidification potential (TAP)
dataset name                                       dataset reference product         dataset unit region                                                                                                          
biodiesel production, via transesterification, ... biodiesel, from palm oil          kilogram     IND                                              0.001617                                                       
                                                                                                  LAM                                              0.001534                                                       
                                                                                                  MEA                                              0.001690                                                       
                                                                                                  OAS                                              0.001581                                                       
                                                                                                  SSA                                              0.001646                                                       
...                                                                                                                                                     ...                                                       
wood pellet production                             wood pellet, measured as dry mass kilogram     World                                            0.000064                                                       
                                                                                                  World                                            0.000064                                                       
                                                                                                  World                                            0.000064                                                       
                                                                                                  World                                            0.000064                                                       
                                                                                                  World                                            0.000064                                                       

[964 rows x 1 columns]

In [ ]:
Q = np.random.random((1, 100))
Imat = np.random.random((100, 50))

In [ ]:
(Q @ Imat).sum(axis=-1)

array([1388.29838872])

In [ ]:
import xarray as xr

In [ ]:
methods = list(monetization.keys())

In [ ]:
impacts = xr.DataArray(
    (Q @ Imat).sum(axis=-1),
    coords = {
        "LCIA method": methods
    }
)

In [ ]:
mfs = xr.DataArray(
    np.diag(list(monetization.values())),
    {
        "LCIA method": methods,
        "impact category": methods
    }
)

In [ ]:
(mfs * impacts).sum(dim="LCIA method").to_dataframe(name="cost").reset_index()

,impact category,cost
0,"ReCiPe 2016 v1.03, midpoint (H) - acidificatio...",7316.332509
